# nb10 — AudioSet CNN Featurizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigreGotico/ww-trainer/blob/dev/notebooks/nb10_audioset_featurizer.ipynb)

> **A recipe for training your own general-audio featurizer from scratch — no labels required.**  \
> This notebook trains a compact CNN on AudioSet using *self-supervised contrastive learning*,\
> then exports it to ONNX so it can be used as a drop-in `OnnxFeatureExtractor` for wake-word\
> training (the same role `tinyhubert.onnx` plays in nb08).

---

## What you are building, and why

A **featurizer** turns raw audio into a compact numeric representation that downstream models
learn from. The quickstart uses hand-designed MFCC features; nb08 uses a featurizer distilled
from speech-only HuBERT. This notebook trains a **general-audio** featurizer on **AudioSet** —
so its features describe *all kinds of sound*, not just speech.

- **AudioSet** is a large public dataset from Google: ~2 million 10-second YouTube clips
  spanning ~600 sound classes (speech, music, animals, vehicles, household noises, …). Here
  it is *streamed* from HuggingFace, so nothing is downloaded to disk in bulk.
- The result is published, when ready, to `TigreGotico/onnx-feature-extractors` on
  HuggingFace for reuse via `OnnxFeatureExtractor`. **This notebook is the training recipe;
  treat any published model as a separate, future artefact.**

The architecture is deliberately small and fully ONNX-exportable: a 4-block residual CNN over
log-mel spectrograms, a frequency pool, a GRU for temporal context, and a 256-dimensional
output — about 3 million parameters.

---

## Self-supervised contrastive learning, explained

AudioSet's labels are noisy and weak, so this recipe ignores them and learns
**self-supervised** — purely from the audio itself.

- **Contrastive learning** trains a model to pull *related* inputs together and push
  *unrelated* inputs apart in embedding space. No class labels needed.
- **Positive pair** — here, two differently *augmented* crops of the *same* clip. They should
  end up with similar embeddings (same underlying sound).
- **Negatives** — every other clip in the batch. Their embeddings should be pushed away.
- **NT-Xent loss** (Normalized Temperature-scaled Cross-Entropy, the SimCLR loss) is the
  objective that enforces this: for each clip, its augmented twin should be the most similar
  item in the batch, and everything else less so. The *temperature* controls how sharply the
  model distinguishes near-misses (lower = stricter).

The intuition: by learning to recognise "these two noisy views are the same sound", the model
is forced to capture the essential structure of audio — exactly the kind of representation a
downstream classifier can build on.

---

## What this notebook does

| Cell | Step | What happens |
|------|------|-------------|
| 1 | **Configure** | Hyperparameters, platform detection, HF token |
| 2 | **Install** | Install `ww_trainer`, `datasets`, `huggingface_hub` |
| 3 | **Define model + loss** | `AudioSetCNNExtractor` and `nt_xent_loss` |
| 4 | **Streaming loader** | Stream AudioSet, emit augmented positive pairs |
| 5 | **Train** | Contrastive training loop with checkpoints |
| 6 | **Loss curve** | Plot per-step and per-epoch NT-Xent loss |
| 7 | **Export ONNX** | Export the featurizer (+ INT8 quantized variant) |
| 8 | **Verify ONNX** | Round-trip check via `OnnxFeatureExtractor` |
| 9 | **Wake-word test** | Optional: use the featurizer to train a quick detector |
| 10 | **Publish** | Optional: upload the ONNX to HuggingFace |

**Designed for the Kaggle free tier** (T4 GPU, 13 GB RAM, ~30 GB disk). The defaults
(10 epochs × 500 steps) finish in a reasonable session; raise `EPOCHS`/`STEPS_PER_EPOCH` for
a stronger featurizer if you have the time budget.

---

## Output files

```
audioset_feat/
├── checkpoint_epoch{N}.pt              ← per-epoch PyTorch checkpoints
├── best.pt                            ← lowest-loss checkpoint
├── audioset_cnn_extractor.onnx        ← exported featurizer  ← the deliverable
├── audioset_cnn_extractor_int8.onnx   ← INT8-quantized variant (smaller, faster)
└── loss_curve.png                     ← training loss plot
```

The exported featurizer takes raw 16 kHz audio `[B, T]` and outputs `[B, T'', 256]` — ready
to feed into wake-word training exactly like any other `OnnxFeatureExtractor`.

## Cell 1 — Configuration and platform detection

Sets every training hyperparameter and detects whether you are on Kaggle or Colab so the
output directory and secrets are wired correctly.

Key knobs (all overridable via environment variables):

- `EPOCHS`, `STEPS_PER_EPOCH` — total training work. The defaults (10 × 500 = 5000 steps)
  are a Kaggle-session-sized recipe; increase both for a stronger featurizer.
- `BATCH_SIZE` — number of clips per step. In contrastive learning, **bigger batches mean
  more negatives per positive**, which usually helps; reduce it if you hit out-of-memory.
- `LR` — learning rate for the Adam optimiser.
- `TEMPERATURE` — the NT-Xent temperature (default `0.07`); lower makes the model
  distinguish near-misses more sharply.
- `PUSH_TO_HF`, `HF_REPO`, `HF_TOKEN` — optional upload settings for Cell 10. On Kaggle the
  token can come from a Secret named `HF_TOKEN`.

The cell auto-selects `cuda` when a GPU is present (recommended) and falls back to `cpu`.

## Cell 2 — Install dependencies

Installs `ww_trainer` (for the `BaseExtractor` / `OnnxFeatureExtractor` building blocks and
the optional wake-word test in Cell 9), the HuggingFace `datasets` library (for streaming
AudioSet), and `huggingface_hub` (for the optional upload in Cell 10).

This is quick and safe to re-run; pip skips anything already present.

## Cell 3 — Define the model and the contrastive loss

This cell defines the featurizer architecture and the NT-Xent loss, then runs a tiny
self-test (a forward pass on dummy audio) to confirm the output shape and parameter count.

**`AudioSetCNNExtractor`** (subclass of `ww_trainer.feats.BaseExtractor`) processes audio in
six stages:

1. **Log-mel spectrogram** — raw waveform → time-frequency energy image, computed with pure
   PyTorch ops (`torch.stft` + a mel filterbank held as a buffer) so the whole thing is
   ONNX-exportable.
2. **Residual CNN backbone** — four `CnnBlock`s (Conv → BatchNorm → GELU, with skip
   connections) that progressively downsample and deepen the feature maps.
3. **Frequency pooling** — collapse the frequency axis, keeping the time axis.
4. **GRU** — adds temporal context across frames.
5. **LayerNorm** — stabilises the 256-dim output.

The mel filterbank and Hann window are stored with `register_buffer`, so they are part of
the model (saved/loaded and exported) but are not trainable parameters.

**`nt_xent_loss(z_i, z_j, temperature)`** implements the contrastive objective: given the
two augmented views of each clip in a batch, it L2-normalises the embeddings, builds the
cosine-similarity matrix, masks out self-similarity, and uses cross-entropy to reward each
clip for being most similar to its own augmented twin. (Note: this is a standalone helper;
it is *not* the same class as `ww_trainer.loss.NTXentLoss`, which serves a different,
supervised role.)

> The `assert out.shape[-1] == 256` line is a guard — if it fails, the architecture and the
> downstream `feature_dim` have drifted out of sync.

## Cell 4 — Streaming dataset that emits positive pairs

This builds the data pipeline that feeds contrastive training. Crucially, AudioSet is
**streamed** from HuggingFace (`streaming=True`) — clips arrive one at a time over the
network and nothing is bulk-downloaded, which is what keeps this within the Kaggle disk
budget.

`AudioSetStreamDataset` yields a **positive pair** for each clip: it takes one AudioSet clip
and applies `augment_waveform` twice, independently, producing two different "views" of the
same sound (`crop_i`, `crop_j`). The augmentations are:

1. **Random crop / pad** to a fixed 3-second length.
2. **Random gain** (±6 dB) — volume jitter.
3. **Gaussian noise** at ~30 dB SNR — mild additive noise.

Two views of the same clip form the positive pair the NT-Xent loss pulls together; views of
*different* clips in the batch are the negatives it pushes apart. Corrupt or too-short clips
are silently skipped so a few bad examples never crash the stream.

`max_samples = STEPS_PER_EPOCH * BATCH_SIZE` caps how many clips are pulled per epoch.

> **SNR** (Signal-to-Noise Ratio) measures how loud the signal is relative to the added
> noise, in decibels — higher means cleaner. 30 dB is light noise.

## Cell 5 — Training loop

The contrastive training loop. For each step it runs both augmented views through the model,
**mean-pools each clip's frames into a single 256-dim embedding** (`feats.mean(dim=1)`), and
applies `nt_xent_loss`. Standard machinery surrounds it: the Adam optimiser, a cosine
learning-rate schedule, gradient clipping, and — on GPU — automatic mixed precision (AMP)
for speed and lower memory.

After every epoch it saves a checkpoint and, when the epoch's mean loss improves, updates
`best.pt`. A fresh streaming dataset is built each epoch so the stream restarts cleanly.

**What "good" looks like:** the NT-Xent loss should fall steadily and then flatten. There is
no accuracy number here — this is self-supervised, so loss is the only training signal. The
real test of quality comes later: the ONNX round-trip (Cell 8) and the downstream wake-word
test (Cell 9).

> **Expected runtime:** roughly proportional to `EPOCHS × STEPS_PER_EPOCH`. On a Kaggle T4,
> the defaults complete within a normal session. Network speed (streaming AudioSet) is often
> the real bottleneck, not the GPU.

## Cell 6 — Loss curve

Plots the training loss two ways: the fine-grained per-step curve (sampled every 50 steps,
left) and the per-epoch mean (right), with the best loss marked.

**How to read it:** a healthy contrastive run shows the loss dropping quickly at first and
then levelling off. A flat line from the start usually means the learning rate is too low or
the data stream stalled; a loss that jumps around wildly suggests the learning rate is too
high or the batch is too small (too few negatives). The epoch curve flattening near the best
value is the signal that more epochs would yield diminishing returns.

## Cell 7 — Export to ONNX

Reloads the best checkpoint on CPU and exports the featurizer to ONNX. Two artefacts are
produced when `quantize=True`: the full-precision `audioset_cnn_extractor.onnx` and a smaller
INT8-quantized variant (`..._int8.onnx`).

The export uses the **dynamo** path (`export_to_onnx` forces `dynamo=True`). This matters
because the model uses `torch.stft`, which the older TorchScript exporter does not handle
cleanly; the newer dynamo exporter does.

> **INT8 quantization** stores weights as 8-bit integers instead of 32-bit floats, roughly
> quartering the file size and often speeding up CPU inference, at a small accuracy cost.
> Both variants are kept so you can choose the size/accuracy trade-off later.

## Cell 8 — Verify the ONNX round-trip

Loads the exported ONNX back through `OnnxFeatureExtractor` — the exact class downstream
wake-word training uses — and checks it behaves correctly on three inputs: silence, a batch
of 3-second clips, and a longer 10-second clip.

The assertions confirm two things that matter for real use:

- **`feature_dim == 256`** — the contract downstream code relies on.
- **Dynamic length** — the model accepts variable-length audio (different `T`) and variable
  batch sizes, producing `[B, T'', 256]` outputs. This is essential because real audio is
  never a fixed length.

If every check passes, the featurizer is a valid drop-in `OnnxFeatureExtractor`.

## Cell 9 — Quick wake-word test (optional)

The ultimate sanity check: does this featurizer actually work for the downstream task it was
built for? This cell trains a quick wake-word detector (just 5 epochs) using the freshly
exported ONNX as the featurizer, via `train_from_wakeword(..., featurizer_type="onnx",
featurizer_path=...)`.

It is wrapped in `try/except` and gated by `SKIP_WW_EVAL` because it needs the `[datagen]`
extras and network access to build a small dataset. If it fails or is skipped, that does not
invalidate the featurizer — the ONNX is still valid; this is only an end-to-end convenience
check. Set `SKIP_WW_EVAL = True` to skip it in an offline environment.

> For a *proper* evaluation of the featurizer, train a full wake-word model with it (see the
> quickstart) or compare it against other featurizers in `nb09_ablation.ipynb`.

## Cell 10 — Publish to HuggingFace (optional)

Uploads the featurizer ONNX (and, if present, the INT8 variant and the loss-curve plot) to a
HuggingFace model repo. This step only runs when `PUSH_TO_HF=true` **and** an `HF_TOKEN` is
available; otherwise it just lists the local artefacts and does nothing remote.

The intended destination is `TigreGotico/onnx-feature-extractors`, the shared featurizer hub
referenced in the project roadmap. Publishing makes the model reusable across projects via
`OnnxFeatureExtractor` (and, in future, a `from_pretrained`-style loader).

### Where to go next

- **Use your featurizer.** Point any wake-word training run at the exported ONNX
  (`featurizer_type="onnx"`, `featurizer_path=...`) — see Cell 9 for the call shape.
- **Benchmark it.** `ww_trainer.benchmark` reports latency and the **real-time factor (RTF)**
  — inference time divided by audio duration. RTF below ~0.1 is comfortable for always-on
  detection; see `docs/guides/evaluation.md`.
- **Compare against alternatives.** `nb09_ablation.ipynb` slots different featurizers into
  the same grid so you can see whether this general-audio encoder beats MFCC, filterbank, or
  the distilled-HuBERT featurizer from `nb08_wakehubert.ipynb` for your wake word.

In [ ]:
# Cell 1 — Config + platform detection
import os

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB  = "google.colab" in str(globals())

WAKE_WORD       = "hey jarvis"           # not used for training, used for quick eval
OUTPUT_DIR      = os.environ.get("OUTPUT_DIR", "/kaggle/working/audioset_feat" if ON_KAGGLE else "./audioset_feat")
EPOCHS          = int(os.environ.get("EPOCHS", "10"))
BATCH_SIZE      = int(os.environ.get("BATCH_SIZE", "64"))
STEPS_PER_EPOCH = int(os.environ.get("STEPS_PER_EPOCH", "500"))
LR              = float(os.environ.get("LR", "3e-4"))
TEMPERATURE     = float(os.environ.get("TEMPERATURE", "0.07"))
PUSH_TO_HF      = os.environ.get("PUSH_TO_HF", "false").lower() == "true"
HF_REPO         = os.environ.get("HF_REPO", "TigreGotico/onnx-feature-extractors")
HF_TOKEN        = os.environ.get("HF_TOKEN", "")

# Kaggle secret fallback
if ON_KAGGLE and not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

import pathlib
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  |  Output: {OUTPUT_DIR}")
print(f"Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  Steps/epoch: {STEPS_PER_EPOCH}  |  LR: {LR}")

In [ ]:
# Cell 2 — Install dependencies
import subprocess, sys

pkgs = ["ww_trainer[datagen]", "datasets", "huggingface_hub"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
print("Dependencies installed.")

In [ ]:
# Cell 3 — Define AudioSetCNNExtractor and NT-Xent loss
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from ww_trainer.feats import BaseExtractor


def _build_mel_filterbank_nb(n_mels: int, n_fft: int, sample_rate: int,
                              f_min: float = 0.0, f_max: float = None) -> torch.Tensor:
    """Build triangular mel filterbank matrix — pure PyTorch, ONNX-safe.
    Returns [n_mels, n_fft // 2 + 1].
    """
    if f_max is None:
        f_max = sample_rate / 2.0

    def hz_to_mel(f):
        return 2595.0 * torch.log10(1.0 + f / 700.0)

    def mel_to_hz(m):
        return 700.0 * (10 ** (m / 2595.0) - 1.0)

    m_min = hz_to_mel(torch.tensor(float(f_min)))
    m_max = hz_to_mel(torch.tensor(float(f_max)))
    m_points = torch.linspace(m_min, m_max, n_mels + 2)
    f_points = mel_to_hz(m_points)
    bins = torch.floor((n_fft + 1) * f_points / sample_rate).long()

    fb = torch.zeros(n_mels, n_fft // 2 + 1)
    for m in range(1, n_mels + 1):
        f_m_minus = bins[m - 1].item()
        f_m       = bins[m].item()
        f_m_plus  = bins[m + 1].item()
        for k in range(f_m_minus, f_m):
            if f_m > f_m_minus:
                fb[m - 1, k] = (k - f_m_minus) / (f_m - f_m_minus)
        for k in range(f_m, f_m_plus):
            if f_m_plus > f_m:
                fb[m - 1, k] = (f_m_plus - k) / (f_m_plus - f_m)
    return fb


class CnnBlock(nn.Module):
    """Residual CNN block: Conv2d -> BN -> GELU -> Conv2d -> BN + skip connection."""

    def __init__(self, in_ch: int, out_ch: int, kernel_size=(3, 3), stride=(1, 1)):
        super().__init__()
        pad = (kernel_size[0] // 2, kernel_size[1] // 2)
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.act   = nn.GELU()
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size, stride=1, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        # Skip projection when shape changes
        if in_ch != out_ch or stride != (1, 1):
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.skip(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act(out + residual)


class AudioSetCNNExtractor(BaseExtractor):
    """Compact contrastively-trained audio feature extractor.

    Input:  [B, T]  raw 16 kHz float32
    Output: [B, T'', 256]  (feature_dim = 256)

    All mel filterbank computation is done via register_buffer tensors +
    pure torch ops so the model is fully ONNX-exportable via dynamo.
    """

    N_MELS    = 128
    N_FFT     = 1024
    HOP       = 320   # 20 ms at 16 kHz
    WIN       = 1024  # ~64 ms at 16 kHz
    FEAT_DIM  = 256

    def __init__(self, sample_rate: int = 16000, device: str = "auto"):
        super().__init__(sample_rate=sample_rate, device=device)

        # --- Mel filterbank (registered buffer — not a parameter) ---
        mel_fb = _build_mel_filterbank_nb(
            n_mels=self.N_MELS,
            n_fft=self.N_FFT,
            sample_rate=sample_rate,
            f_min=0.0,
            f_max=sample_rate / 2.0,
        )  # [128, 513]
        self.register_buffer("mel_fb", mel_fb)

        # Hann window (registered buffer)
        self.register_buffer("hann_win", torch.hann_window(self.WIN))

        # --- CNN backbone (4 residual blocks) ---
        self.cnn = nn.Sequential(
            CnnBlock(1,   32,  kernel_size=(3, 3), stride=(2, 2)),   # block1
            CnnBlock(32,  64,  kernel_size=(3, 3), stride=(2, 2)),   # block2
            CnnBlock(64,  128, kernel_size=(3, 3), stride=(2, 2)),   # block3
            CnnBlock(128, 256, kernel_size=(3, 3), stride=(1, 1)),   # block4
        )

        # Collapse frequency axis while keeping time
        self.pool = nn.AdaptiveAvgPool2d((1, None))  # [B, 256, 1, T'']

        # Temporal context aggregation
        self.gru  = nn.GRU(256, 256, num_layers=1, batch_first=True)
        self.norm = nn.LayerNorm(256)

    @property
    def feature_dim(self) -> int:
        return self.FEAT_DIM

    def _log_mel(self, wavs: torch.Tensor) -> torch.Tensor:
        """Compute log-mel spectrogram from raw waveform [B, T] -> [B, T', N_MELS].

        Uses torch.stft with return_complex=False (real/imag stack) for full
        ONNX compatibility when exported via the dynamo path.
        """
        B = wavs.shape[0]

        # Ensure minimum length for STFT
        if wavs.shape[-1] < self.N_FFT:
            wavs = F.pad(wavs, (0, self.N_FFT - wavs.shape[-1]))

        # Flatten batch for stft (operates on 1D or 2D inputs)
        # torch.stft expects [T] or [B, T]; use batched form directly
        stft_out = torch.stft(
            wavs,
            n_fft=self.N_FFT,
            hop_length=self.HOP,
            win_length=self.WIN,
            window=self.hann_win,
            return_complex=False,  # ONNX-safe: returns [B, F, T', 2]
            center=False,
        )  # [B, F, T', 2]

        real = stft_out[..., 0]  # [B, F, T']
        imag = stft_out[..., 1]  # [B, F, T']
        power = real.pow(2) + imag.pow(2)  # [B, F, T']

        # Apply mel filterbank: [n_mels, F] x [B, F, T'] -> [B, n_mels, T']
        mel_spec = torch.matmul(self.mel_fb, power)  # [B, N_MELS, T']
        log_mel  = torch.log(mel_spec + 1e-10)       # [B, N_MELS, T']

        return log_mel.transpose(1, 2)  # [B, T', N_MELS]

    def forward(self, wavs: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            wavs: [B, T] raw 16 kHz float32 waveform.

        Returns:
            [B, T'', 256] feature tensor.
        """
        # 1. Log-mel: [B, T', 128]
        log_mel = self._log_mel(wavs)  # [B, T', 128]

        # 2. Reshape to image-like [B, 1, 128, T'] (freq as height, time as width)
        x = log_mel.transpose(1, 2).unsqueeze(1)  # [B, 1, 128, T']

        # 3. CNN backbone
        x = self.cnn(x)  # [B, 256, H'', T'']

        # 4. Pool over frequency axis: [B, 256, 1, T'']
        x = self.pool(x)  # [B, 256, 1, T'']
        x = x.squeeze(2)  # [B, 256, T'']
        x = x.transpose(1, 2)  # [B, T'', 256]

        # 5. GRU temporal context
        x, _ = self.gru(x)   # [B, T'', 256]

        # 6. LayerNorm
        x = self.norm(x)  # [B, T'', 256]

        return x

    def export_to_onnx(self, out: str, quantize: bool = False,
                       dynamo: bool = True, metadata: dict = None) -> None:
        """Export to ONNX via the dynamo path (required for torch.stft compatibility)."""
        super().export_to_onnx(out, quantize=quantize, dynamo=True, metadata=metadata)


# --- NT-Xent contrastive loss ---

def nt_xent_loss(z_i: torch.Tensor, z_j: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    """NT-Xent (Normalized Temperature-scaled Cross Entropy) loss.

    Args:
        z_i: [B, D] L2-normalised embeddings for view i.
        z_j: [B, D] L2-normalised embeddings for view j.
        temperature: Softmax temperature (default 0.07).

    Returns:
        Scalar loss tensor.
    """
    B = z_i.shape[0]

    # L2 normalise
    z_i = F.normalize(z_i, dim=-1)
    z_j = F.normalize(z_j, dim=-1)

    # Concatenate: [2B, D]
    z = torch.cat([z_i, z_j], dim=0)

    # Pairwise cosine similarity matrix [2B, 2B]
    sim = torch.mm(z, z.t()) / temperature

    # Mask out self-similarity on diagonal
    mask = torch.eye(2 * B, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(mask, float("-inf"))

    # Positive pair indices: (i, B+i) and (B+i, i)
    labels = torch.cat([torch.arange(B, 2 * B), torch.arange(0, B)], dim=0).to(z.device)

    loss = F.cross_entropy(sim, labels)
    return loss


# Quick sanity-check
model = AudioSetCNNExtractor(sample_rate=16000, device="cpu")
dummy = torch.zeros(2, 48000)  # 2 clips × 3 seconds
out   = model(dummy)
print(f"Model output shape: {tuple(out.shape)}")
print(f"feature_dim: {model.feature_dim}")
assert out.shape[-1] == 256, f"Expected feature_dim=256, got {out.shape[-1]}"
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,} ({total_params / 1e6:.2f}M)")

In [ ]:
# Cell 4 — Dataset streaming loader
import random
from itertools import islice

import torch
import torchaudio
from torch.utils.data import DataLoader, IterableDataset


CROP_SAMPLES = 48000  # 3 seconds at 16 kHz


def augment_waveform(wav: torch.Tensor, crop_samples: int = CROP_SAMPLES) -> torch.Tensor:
    """Apply random augmentations to a 1-D waveform tensor.

    Augmentations:
      1. Random crop / pad to `crop_samples`.
      2. Random gain ±6 dB.
      3. Gaussian noise at ~30 dB SNR.

    All ops are pure torch — no torchaudio transforms.
    """
    T = wav.shape[-1]

    # 1. Random crop or pad
    if T >= crop_samples:
        start = random.randint(0, T - crop_samples)
        wav = wav[start : start + crop_samples]
    else:
        wav = F.pad(wav, (0, crop_samples - T))

    # 2. Random gain ±6 dB  (6 dB = factor of 2)
    gain_db = random.uniform(-6.0, 6.0)
    gain    = 10 ** (gain_db / 20.0)
    wav     = wav * gain

    # 3. Gaussian noise at ~30 dB SNR
    signal_power = wav.pow(2).mean().clamp_min(1e-10)
    noise_power  = signal_power / (10 ** (30.0 / 10.0))
    noise        = torch.randn_like(wav) * noise_power.sqrt()
    wav          = wav + noise

    return wav.float()


class AudioSetStreamDataset(IterableDataset):
    """Streaming AudioSet dataset that yields positive pairs for contrastive learning.

    Each item is a tuple (crop_i, crop_j) — two independently augmented crops
    of the same AudioSet clip, forming a positive pair.
    """

    def __init__(
        self,
        split: str = "unbalanced_train",
        target_sr: int = 16000,
        crop_samples: int = CROP_SAMPLES,
        max_samples: int = None,
    ):
        super().__init__()
        self.split        = split
        self.target_sr    = target_sr
        self.crop_samples = crop_samples
        self.max_samples  = max_samples

    def _make_stream(self):
        from datasets import load_dataset
        # agkphysics/AudioSet provides audio with array + sampling_rate
        ds = load_dataset(
            "agkphysics/AudioSet",
            "unbalanced",
            split=self.split,
            streaming=True,
            trust_remote_code=False,
        )
        return ds

    def __iter__(self):
        ds = self._make_stream()
        stream = islice(ds, self.max_samples) if self.max_samples else iter(ds)

        for example in stream:
            try:
                audio = example["audio"]
                arr   = audio["array"]
                sr    = audio["sampling_rate"]

                # Convert to torch float32 tensor
                if not isinstance(arr, torch.Tensor):
                    wav = torch.tensor(arr, dtype=torch.float32)
                else:
                    wav = arr.float()

                # Flatten to mono
                if wav.ndim == 2:
                    wav = wav.mean(dim=0)
                elif wav.ndim > 2:
                    wav = wav.reshape(-1)

                # Resample to target_sr if needed
                if sr != self.target_sr:
                    wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=self.target_sr)

                # Skip clips that are too short (< 0.5 s)
                if wav.shape[-1] < self.target_sr // 2:
                    continue

                # Produce two augmented crops — positive pair
                crop_i = augment_waveform(wav.clone(), self.crop_samples)
                crop_j = augment_waveform(wav.clone(), self.crop_samples)

                yield crop_i, crop_j

            except Exception:
                # Silently skip corrupt / malformed examples
                continue


# Build dataset and dataloader
max_samples = STEPS_PER_EPOCH * BATCH_SIZE

train_dataset = AudioSetStreamDataset(
    split="unbalanced_train",
    target_sr=16000,
    crop_samples=CROP_SAMPLES,
    max_samples=max_samples,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=(DEVICE == "cuda"),
    drop_last=True,
)

print(f"Dataset configured: max {max_samples:,} samples per epoch, batch size {BATCH_SIZE}")
print("(Streaming — no disk download required)")

In [ ]:
# Cell 5 — Training loop
import time

import torch
import torch.optim as optim

model = AudioSetCNNExtractor(sample_rate=16000, device=DEVICE)
model = model.to(DEVICE)

total_steps = EPOCHS * STEPS_PER_EPOCH
optimizer   = optim.Adam(model.parameters(), lr=LR)
scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=LR * 0.01)

use_amp    = (DEVICE == "cuda")
scaler     = torch.cuda.amp.GradScaler() if use_amp else None

epoch_losses  = []   # mean loss per epoch
step_losses   = []   # loss every 50 steps (for plot)
best_loss     = float("inf")
global_step   = 0

print(f"Starting training — {EPOCHS} epochs × {STEPS_PER_EPOCH} steps | AMP={use_amp}")

for epoch in range(1, EPOCHS + 1):
    model.train()

    # Rebuild a fresh streaming dataset each epoch (streaming resets automatically)
    epoch_dataset = AudioSetStreamDataset(
        split="unbalanced_train",
        target_sr=16000,
        crop_samples=CROP_SAMPLES,
        max_samples=STEPS_PER_EPOCH * BATCH_SIZE,
    )
    epoch_loader = DataLoader(
        epoch_dataset,
        batch_size=BATCH_SIZE,
        num_workers=2,
        pin_memory=(DEVICE == "cuda"),
        drop_last=True,
    )

    running_loss = 0.0
    n_steps      = 0
    t0           = time.time()

    for step, (crop_i, crop_j) in enumerate(epoch_loader, 1):
        crop_i = crop_i.to(DEVICE, non_blocking=True)  # [B, 48000]
        crop_j = crop_j.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            # Forward both views
            feats_i = model(crop_i)  # [B, T'', 256]
            feats_j = model(crop_j)

            # Global average pool over time for a single embedding vector per clip
            z_i = feats_i.mean(dim=1)  # [B, 256]
            z_j = feats_j.mean(dim=1)

            loss = nt_xent_loss(z_i, z_j, temperature=TEMPERATURE)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()

        loss_val      = loss.item()
        running_loss += loss_val
        n_steps      += 1
        global_step  += 1

        if global_step % 50 == 0:
            avg50 = running_loss / n_steps
            step_losses.append((global_step, avg50))
            lr_now = scheduler.get_last_lr()[0]
            print(f"  step {global_step:5d} | loss {loss_val:.4f} | lr {lr_now:.2e}")

        if step >= STEPS_PER_EPOCH:
            break

    epoch_mean = running_loss / max(n_steps, 1)
    epoch_losses.append(epoch_mean)
    elapsed = time.time() - t0
    print(f"Epoch {epoch}/{EPOCHS} | mean_loss={epoch_mean:.4f} | steps={n_steps} | {elapsed:.1f}s")

    # Save checkpoint
    ckpt_path = f"{OUTPUT_DIR}/checkpoint_epoch{epoch}.pt"
    torch.save(model.state_dict(), ckpt_path)
    print(f"  Checkpoint saved: {ckpt_path}")

    # Track best
    if epoch_mean < best_loss:
        best_loss = epoch_mean
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pt")
        print(f"  New best loss: {best_loss:.4f} — saved best.pt")

print(f"\nTraining complete. Best loss: {best_loss:.4f}")

In [ ]:
# Cell 6 — Loss curve plot
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Per-step loss (every 50 steps)
if step_losses:
    steps, losses = zip(*step_losses)
    axes[0].plot(steps, losses, linewidth=1.2, color="steelblue")
    axes[0].set_xlabel("Global step")
    axes[0].set_ylabel("NT-Xent loss")
    axes[0].set_title("Training loss (every 50 steps)")
    axes[0].grid(True, alpha=0.3)

# Per-epoch mean loss
if epoch_losses:
    axes[1].plot(range(1, len(epoch_losses) + 1), epoch_losses,
                 marker="o", linewidth=1.5, color="darkorange")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Mean NT-Xent loss")
    axes[1].set_title("Epoch mean loss")
    axes[1].grid(True, alpha=0.3)
    axes[1].axhline(y=best_loss, linestyle="--", color="red", alpha=0.6, label=f"Best: {best_loss:.4f}")
    axes[1].legend()

plt.tight_layout()
plot_path = f"{OUTPUT_DIR}/loss_curve.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(f"Plot saved: {plot_path}")

In [ ]:
# Cell 7 — ONNX export
import torch

best_pt = f"{OUTPUT_DIR}/best.pt"
onnx_path = f"{OUTPUT_DIR}/audioset_cnn_extractor.onnx"

# Reload best weights on CPU for clean export
model.load_state_dict(torch.load(best_pt, map_location="cpu"))
model.eval().cpu()

print(f"Exporting to ONNX (dynamo path, with INT8 quantization)...")
model.export_to_onnx(onnx_path, quantize=True)
print(f"Exported: {onnx_path}")

# List generated files
import pathlib
for f in sorted(pathlib.Path(OUTPUT_DIR).glob("*.onnx")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}  ({size_mb:.2f} MB)")

In [ ]:
# Cell 8 — Verify with OnnxFeatureExtractor round-trip
from ww_trainer.feats import OnnxFeatureExtractor
import torch
import numpy as np

onnx_path = f"{OUTPUT_DIR}/audioset_cnn_extractor.onnx"
ext = OnnxFeatureExtractor(onnx_path)

# Test with silence
wav = torch.zeros(1, 16000)
feats = ext(wav)
print(f"feature_dim={ext.feature_dim}  output_shape={tuple(feats.shape)}")
assert ext.feature_dim == 256, f"Expected feature_dim=256, got {ext.feature_dim}"

# Test with batch of 3-second clips
wav_batch = torch.randn(4, 48000) * 0.01
feats_batch = ext(wav_batch)
print(f"Batch input [4, 48000] -> output {tuple(feats_batch.shape)}")
assert feats_batch.shape[0] == 4
assert feats_batch.shape[-1] == 256

# Test dynamic length
wav_long = torch.randn(2, 160000) * 0.01  # 10 seconds
feats_long = ext(wav_long)
print(f"Long input [2, 160000] -> output {tuple(feats_long.shape)}")

print("\nAll ONNX round-trip checks passed.")

In [ ]:
# Cell 9 — Quick wake-word training test (optional, ~5 epochs)
# Confirms the exported ONNX featurizer works end-to-end as a feature extractor
# for downstream wake-word training.
#
# NOTE: This cell requires ww_trainer to be installed with [datagen] extras
# and network access to download the dataset. On Kaggle, this works out of the box.
# Skip if running in an offline environment.

SKIP_WW_EVAL = False  # set True to skip this cell

if not SKIP_WW_EVAL:
    try:
        from ww_trainer.quickstart import train_from_wakeword
        import pathlib

        ww_output = f"{OUTPUT_DIR}/ww_test"
        pathlib.Path(ww_output).mkdir(parents=True, exist_ok=True)

        onnx_path = f"{OUTPUT_DIR}/audioset_cnn_extractor.onnx"

        print(f"Running quick WW training test for '{WAKE_WORD}' with ONNX featurizer...")
        print("(5 epochs, reuse_dataset=True)")

        result = train_from_wakeword(
            wake_word=WAKE_WORD,
            output_dir=ww_output,
            epochs=5,
            batch_size=16,
            featurizer_type="onnx",
            featurizer_path=onnx_path,
            reuse_dataset=True,
            tier="small",
        )
        print(f"WW training complete. Result: {result}")

    except Exception as e:
        print(f"WW training test skipped / failed: {e}")
        print("This is non-critical — the featurizer ONNX is valid regardless.")
else:
    print("WW evaluation skipped (SKIP_WW_EVAL=True).")

In [ ]:
# Cell 10 — Upload to HuggingFace
import pathlib

onnx_path      = f"{OUTPUT_DIR}/audioset_cnn_extractor.onnx"
onnx_int8_path = f"{OUTPUT_DIR}/audioset_cnn_extractor_int8.onnx"

if PUSH_TO_HF and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)

    print(f"Uploading to https://huggingface.co/{HF_REPO} ...")

    # Main ONNX
    api.upload_file(
        path_or_fileobj=onnx_path,
        path_in_repo="audioset_cnn_extractor.onnx",
        repo_id=HF_REPO,
        repo_type="model",
        commit_message="Add AudioSet CNN featurizer ONNX",
    )
    print(f"Uploaded: audioset_cnn_extractor.onnx")

    # INT8 quantised variant
    if pathlib.Path(onnx_int8_path).exists():
        api.upload_file(
            path_or_fileobj=onnx_int8_path,
            path_in_repo="audioset_cnn_extractor_int8.onnx",
            repo_id=HF_REPO,
            repo_type="model",
            commit_message="Add AudioSet CNN featurizer INT8 ONNX",
        )
        print(f"Uploaded: audioset_cnn_extractor_int8.onnx")
    else:
        print(f"INT8 variant not found at {onnx_int8_path} — skipping.")

    # Also upload the loss curve plot if present
    plot_path = f"{OUTPUT_DIR}/loss_curve.png"
    if pathlib.Path(plot_path).exists():
        api.upload_file(
            path_or_fileobj=plot_path,
            path_in_repo="audioset_cnn_extractor_loss_curve.png",
            repo_id=HF_REPO,
            repo_type="model",
            commit_message="Add AudioSet CNN featurizer training loss curve",
        )
        print("Uploaded: loss curve plot")

    print(f"\nDone. View at: https://huggingface.co/{HF_REPO}")
else:
    print("Set PUSH_TO_HF=true and HF_TOKEN to upload.")
    print(f"Local artifacts in: {OUTPUT_DIR}")
    for f in sorted(pathlib.Path(OUTPUT_DIR).iterdir()):
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.name}  ({size_mb:.2f} MB)")